# Pytorch

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset, random_split
from sklearn.datasets import fetch_california_housing


data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['MedHouseVal'] = data.target
df.to_csv('housing.csv', index=False)
print("Saved housing.csv to your local folder!")

class CaliforniaHousingDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.features = self.data_frame.iloc[:, :-1].values
        self.targets = self.data_frame.iloc[:,-1].values
        self.transform = transform
    def __len__(self):
        return len(self.data_frame)
    
    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.targets[idx]
        
        # Apply the feature transform if it exists
        if self.transform:
            x = self.transform(x)
            
        y = torch.tensor([y], dtype=torch.float32)
        
        return x, y

Saved housing.csv to your local folder!


In [2]:
class ToFloat32Tensor:
    def __call__(self, sample):
        # Converts a numpy array to a float32 tensor
        return torch.tensor(sample, dtype=torch.float32)

my_transforms = transforms.Compose([
    ToFloat32Tensor()
])
full_dataset = CaliforniaHousingDataset('housing.csv', transform = my_transforms)

total_size = len(full_dataset)
train_size = int(0.70 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size  

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, 
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42) # Ensures reproducibility
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [24]:
class HousingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # 1. The Internal Scaler! (8 is the number of input features)
            nn.BatchNorm1d(8), 
            
            # 2. The rest of the network
            nn.Linear(8, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)
    
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HousingModel().to(device)

In [25]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
epochs = 100

for epoch in range(epochs):
    
    # -----------------------
    # TRAINING
    # -----------------------
    model.train() # CRITICAL: Tells BatchNorm to measure current batch stats
    train_loss = 0.0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    # -----------------------
    # EVALUATION
    # -----------------------
    model.eval() # CRITICAL: Tells BatchNorm to freeze and use its moving average
    val_loss = 0.0
    
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            outputs = model(batch_x)
            val_loss += criterion(outputs, batch_y).item()
            
    # Calculate average losses for printing
    avg_train = train_loss / len(train_loader)
    avg_val = val_loss / len(val_loader)
    
    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss (MSE): {avg_train:.4f} | Val Loss (MSE): {avg_val:.4f}")

Epoch 01/100 | Train Loss (MSE): 0.6210 | Test Loss (MSE): 0.6480
Epoch 02/100 | Train Loss (MSE): 0.4377 | Test Loss (MSE): 3.0170
Epoch 03/100 | Train Loss (MSE): 0.4157 | Test Loss (MSE): 1.5667
Epoch 04/100 | Train Loss (MSE): 0.3968 | Test Loss (MSE): 0.9562
Epoch 05/100 | Train Loss (MSE): 0.4130 | Test Loss (MSE): 0.4797
Epoch 06/100 | Train Loss (MSE): 0.3911 | Test Loss (MSE): 0.6428
Epoch 07/100 | Train Loss (MSE): 0.4069 | Test Loss (MSE): 0.7200
Epoch 08/100 | Train Loss (MSE): 0.3962 | Test Loss (MSE): 1.2902
Epoch 09/100 | Train Loss (MSE): 0.3869 | Test Loss (MSE): 0.4605
Epoch 10/100 | Train Loss (MSE): 0.3950 | Test Loss (MSE): 1.3175
Epoch 11/100 | Train Loss (MSE): 0.3813 | Test Loss (MSE): 2.3503
Epoch 12/100 | Train Loss (MSE): 0.3776 | Test Loss (MSE): 0.9578
Epoch 13/100 | Train Loss (MSE): 0.3815 | Test Loss (MSE): 1.1908
Epoch 14/100 | Train Loss (MSE): 0.3853 | Test Loss (MSE): 1.0776
Epoch 15/100 | Train Loss (MSE): 0.3796 | Test Loss (MSE): 0.4695
Epoch 16/1

In [27]:
# ==========================================
# 6. FINAL EVALUATION ON TEST DATA
# ==========================================
print("\n--- Training Complete. Running Final Test ---")

# 1. Put model in read-only mode
model.eval() 
test_loss = 0.0

# 2. Disable gradient tracking
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        # 3. Make predictions
        outputs = model(batch_x)
        
        # 4. Calculate loss
        test_loss += criterion(outputs, batch_y).item()
        
# 5. Average the loss
avg_test = test_loss / len(test_loader)

print(f"Final Test Loss (MSE): {avg_test:.4f}")


--- Training Complete. Running Final Test ---
Final Test Loss (MSE): 0.4522


# Lighting + Optuna
Data is already loaded 

In [3]:
import lightning as L
import optuna
import logging
import json
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.loggers import TensorBoardLogger
import matplotlib.pyplot as plt

logging.getLogger("lightning.pytorch.utilities.rank_zero").setLevel(logging.WARNING)

class OptunaHousingModule(L.LightningModule):
    def __init__(self, hidden_neurons, lr, dropout_rate, weight_decay):
        super().__init__()
        # Save all passed arguments into self.hparams for easy logging
        self.save_hyperparameters()
        self.lr = lr
        self.weight_decay = weight_decay
        # 1. NEW: Create pure Python lists to hold the history in RAM
        self.train_history = []
        self.val_history = []
        
        self.net = nn.Sequential(
            nn.BatchNorm1d(8),
            nn.Linear(8, hidden_neurons),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_neurons, 1)
        )
        
        self.criterion = nn.MSELoss()
    
    def forward(self, x):
        return self.net(x)
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        preds = self(x)
        loss = self.criterion(preds, y)
        self.log("train_loss", loss, on_step=False, on_epoch=True)
        return loss 
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        loss = self.criterion(self(x), y)
        # Log the validation loss with a specific name ("val_loss"). 
        # This is critical because Optuna will search for this exact name later.
        self.log("val_loss", loss, on_step=False, on_epoch=True) 
        return loss
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        loss = self.criterion(self(x), y)
        self.log("test_loss", loss) 
        return loss
    
    def on_train_epoch_end(self):
        # Reach into Lightning's internal memory and grab the averaged loss
        current_loss = self.trainer.callback_metrics.get("train_loss")
        if current_loss is not None:
            # Append the pure number (.item()) to our list
            self.train_history.append(current_loss.item())

    def on_validation_epoch_end(self):
        current_loss = self.trainer.callback_metrics.get("val_loss")
        if current_loss is not None:
            self.val_history.append(current_loss.item())
            
    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)

def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log = True)   
    hidden_neurons = trial.suggest_int("hidden_neurons", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    
    model = OptunaHousingModule(
        hidden_neurons=hidden_neurons,
        lr = lr,
        dropout_rate=dropout_rate,
        weight_decay=weight_decay
    )
    
    trainer = L.Trainer(
        max_epochs=10,               # Train for 10 epochs max for this quick test
        accelerator="auto",          # Automatically use a GPU if one is available
        enable_progress_bar=False,   # Turn off the progress bar so we don't spam the terminal
        logger=False,                # Turn off default logging for this trial
        enable_model_summary=False   # Don't print the network architecture every time
    )
    
    # Start the training process for this trial, passing in the train and val dataloaders
    trainer.fit(model, train_loader, val_loader)
    
    # After 10 epochs, reach into the Trainer's memory, grab the final "val_loss", and return it.
    # Optuna will use this number to judge if the hyperparameters it picked were good or bad.
    return trainer.callback_metrics["val_loss"].item()
    

W0420 18:02:41.499000 7380 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\YeXiaoJun\anaconda3\envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
if __name__ == "__main__":
    print("--- 1. Starting Optuna Hyperparameter Search ---")
    # Create an Optuna study object. "minimize" means we want the lowest possible validation loss.
    study = optuna.create_study(direction="minimize")
    # Start the search! Run the 'objective' function 5 distinct times with different parameters.
    study.optimize(objective, n_trials=5) 
    
    print("\n--- 2. Optimization Finished ---")
    # Extract the absolute best learning rate Optuna found
    best_lr = study.best_params["lr"]
    # Extract the absolute best hidden neuron size Optuna found
    best_hidden = study.best_params["hidden_neurons"]
    best_dropout = study.best_params["dropout_rate"] 
    best_wd = study.best_params["weight_decay"]      
    # Print the winning numbers to the terminal
    #print(f"Best LR: {best_lr:.5f} | Best Hidden: {best_hidden}")
    #print(f"Best Dropout: {best_dropout:.3f} | Best Weight Decay: {best_wd:.5f}")
    
    print("Number of finished trials: {}".format(len(study.trials)))

    print("Best trial:")    
    trial = study.best_trial

    print("  Value: {}".format(trial.value))

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))
    
    print("\n--- 3. Training Final Model with Best Parameters ---")
    # Create a brand new, fresh model using ONLY the winning parameters
    final_model = OptunaHousingModule(
        hidden_neurons=best_hidden, 
        lr=best_lr,
        dropout_rate=best_dropout,
        weight_decay=best_wd
    )
    
    # NEW: Create the Checkpoint Callback
    checkpoint_callback = ModelCheckpoint(
        monitor="val_loss",          # Watch the validation loss
        dirpath="saved_models/",     # Folder to save the weights in
        filename="best-housing-model", # Name of the file
        save_top_k=1,                # Only keep the absolute best 1 model
        mode="min"                   # "min" because we want the lowest loss
    )
    
    # Initialize the Logger
    # csv_logger = CSVLogger(save_dir="logs/", name="final_model_logs")
    tb_logger = TensorBoardLogger(save_dir="tb_logs/", name="my_housing_model")
    # Add the logger to your final Trainer
    # Create a new Trainer for our final run. We leave progress bars ON this time so we can watch it.
    final_trainer = L.Trainer(max_epochs=15, accelerator="auto", callbacks=[checkpoint_callback], logger=tb_logger) # or csv_logger
    # Train the final model
    final_trainer.fit(final_model, train_loader, val_loader)
    
    print("\n--- Raw Training History ---")
    print("Train Losses:", final_model.train_history)
    print("Val Losses:  ", final_model.val_history)
    
    # Or print them side-by-side cleanly:
    print("\nEpoch | Train Loss | Val Loss")
    for i in range(len(final_model.train_history)):
        t_loss = final_model.train_history[i]
        v_loss = final_model.val_history[i]
        print(f"{i:5d} | {t_loss:.4f}     | {v_loss:.4f}")
    
    # ---------------------------------------------------------
    # Plotting the Learning Curves Using CsvLogger
    # ---------------------------------------------------------
    # print("\n--- Plotting Error Curves ---")
    # Lightning saves the metrics to a physical CSV file in your log directory
    # metrics_file = f"{csv_logger.log_dir}/metrics.csv"
    # metrics_df = pd.read_csv(metrics_file)
    
    # Separate the training and validation data
    # (Lightning logs them on separate rows, so we drop the empty spaces)
    # train_loss = metrics_df[['epoch', 'train_loss']].dropna()
    # val_loss = metrics_df[['epoch', 'val_loss']].dropna()

    # Plot the lines
    # plt.figure(figsize=(8, 5))
    # plt.plot(train_loss['epoch'], train_loss['train_loss'], label='Training Loss', color='blue')
    # plt.plot(val_loss['epoch'], val_loss['val_loss'], label='Validation Loss', color='orange')
    
    # Format the graph
    # plt.title('Model Error During Training')
    # plt.xlabel('Epoch')
    # plt.ylabel('Mean Squared Error (Loss)')
    # plt.legend()
    # plt.grid(True)
    
    # Display the graph (or use plt.savefig('loss_curve.png') to save it)
    # plt.show()
    
    print("\n--- 4. Final Evaluation on Test Set ---")
    # Tell the trainer to evaluate the model using the test_loader (data it has never seen).
    # This automatically calls the test_step() function we defined in the class.
    final_trainer.test(final_model, dataloaders=test_loader, ckpt_path="best")
    print(f"Model saved to: {checkpoint_callback.best_model_path}")

[I 2026-04-20 18:02:43,429] A new study created in memory with name: no-name-5f7854aa-fe06-4caa-ad88-4635a9b63f69
c:\Users\YeXiaoJun\anaconda3\envs\torch_env\lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory c:\Users\YeXiaoJun\Desktop\Second Semester\Deep Learning\torch\housing_dataset\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


--- 1. Starting Optuna Hyperparameter Search ---


c:\Users\YeXiaoJun\anaconda3\envs\torch_env\lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\YeXiaoJun\anaconda3\envs\torch_env\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
c:\Users\YeXiaoJun\anaconda3\envs\torch_env\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
[I 2026-04-20 18:02:51,745] Trial 0 finished with value: 0.4593978524208069 and parameters: {'lr': 0.005965323287515437, 'hidden_neu


--- 2. Optimization Finished ---
Number of finished trials: 10
Best trial:
  Value: 0.3796258866786957
  Params: 
    lr: 0.0053505686214163515
    hidden_neurons: 44
    dropout_rate: 0.3157800043353651
    weight_decay: 0.00517001445735742

--- 3. Training Final Model with Best Parameters ---
Epoch 14: 100%|██████████| 226/226 [00:01<00:00, 188.58it/s, v_num=0]       

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]




--- Raw Training History ---
Train Losses: [1.049112319946289, 0.551264226436615, 0.5183283090591431, 0.4881298542022705, 0.48968303203582764, 0.47748446464538574, 0.46630629897117615, 0.4679151773452759, 0.45093074440956116, 0.4573269486427307, 0.4536418616771698, 0.45776715874671936, 0.4482825994491577, 0.4508555829524994, 0.45362377166748047]
Val Losses:   [2635.735595703125, 1.052836537361145, 0.5256073474884033, 0.9909890294075012, 1.8013770580291748, 1.0631506443023682, 0.5059382915496826, 0.6055304408073425, 0.5015826225280762, 0.5069690346717834, 0.4863266944885254, 0.47506532073020935, 0.5028589367866516, 0.48058849573135376, 0.43652549386024475, 0.3916615843772888]

Epoch | Train Loss | Val Loss
    0 | 1.0491     | 2635.7356
    1 | 0.5513     | 1.0528
    2 | 0.5183     | 0.5256
    3 | 0.4881     | 0.9910
    4 | 0.4897     | 1.8014
    5 | 0.4775     | 1.0632
    6 | 0.4663     | 0.5059
    7 | 0.4679     | 0.6055
    8 | 0.4509     | 0.5016
    9 | 0.4573     | 0.5070


c:\Users\YeXiaoJun\anaconda3\envs\torch_env\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 25/25 [00:00<00:00, 314.08it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           0.4073176980018616
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Model saved to: C:\Users\YeXiaoJun\Desktop\Second Semester\Deep Learning\torch\housing_dataset\saved_models\best-housing-model-v3.ckpt


# How to load saved parameters and hyperparameters

In [1]:
# # 2 Scenarios
# # Scenario 1: Load in the same script

# # a. Get the path to the best model saved by the callback
# best_model_path = checkpoint_callback.best_model_path

# # b. Load the model from that checkpoint
# loaded_model = OptunaHousingModule.load_from_checkpoint(best_model_path)

# # c. (Optional) Set to evaluation mode if you are going to make predictions
# loaded_model.eval()

# print(f"Successfully loaded model from: {best_model_path}")

# # Scenario 2: Loading it in a brand new script later


# # Make sure to import your model class definition!
# from your_model_file import OptunaHousingModule 

# # a. Point to the exact file path
# checkpoint_path = "saved_models/best-housing-model.ckpt"

# # b. Load the model
# loaded_model = OptunaHousingModule.load_from_checkpoint(checkpoint_path)

# # c. Put the model in evaluation mode (turns off dropout)
# loaded_model.eval()

# # 4. Make a prediction (Example)
# # dummy_input = torch.randn(1, num_features) # Replace with real data
# # with torch.no_grad():
# #     prediction = loaded_model(dummy_input)
# #     print(prediction)


